In [ ]:
# conda activate genomic_tools

import os
import sys
import pickle
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from collections import defaultdict

sys.path.append("code")

# from modified_functions import *

In [ ]:
def find_overlapping_cds(exon_start, exon_end, transcript_cds_rows):
    """Return the CDS row that overlaps this exon, or None."""
    for _, cds in transcript_cds_rows.iterrows():
        if cds['end'] >= exon_start and cds['start'] <= exon_end:
            return cds
    return None

def classify_overlap(exon_start, exon_end, cds_row):
    if cds_row['start'] == exon_start and cds_row['end'] == exon_end:
        return "fully_coding"
    elif cds_row['start'] >= exon_start and cds_row['end'] <= exon_end:
        return "partially_coding" # CDS subset of exon (should only happen when part of exon is UTR)
    else:
        return "unexpected"    # one side hangs over — rare
    
def _flanks_match(exons, es, ee, us_intron_start, ds_intron_end):
    left = [x for x in exons if x['end'] < es]
    right = [x for x in exons if x['start'] > ee]
    if not left or not right:
        return False                                  # terminal exon in this transcript
    left_exon = max(left, key=lambda x: x['end'])
    right_exon = min(right, key=lambda x: x['start'])
    return (left_exon['end'] + 1 == us_intron_start) and (right_exon['start'] - 1 == ds_intron_end)

def classify_isoform_detailed(exons, es, ee, us_intron_start, ds_intron_end, strand):
    """Returns (bucket, detail). detail carries the boundary sub-type for
    exon_diff_boundary, else None."""
    exact = overlap = None
    has_up = has_down = exonic_overlap = False
    for ex in exons:
        s, e = ex['start'], ex['end']
        if e < es:
            has_up = True
        if s > ee:
            has_down = True
        if s <= ee and e >= es:
            exonic_overlap = True
            if s == es and e == ee:
                exact = ex
            else:
                overlap = ex
    if exact is not None:
        s, e = exact['start'], exact['end']   # use exact, not last loop var
        bucket = ("compatible" if _flanks_match(exons, es, ee, us_intron_start, ds_intron_end)
                else "exon_diff_junction")
        return bucket, {'start': s, 'end': e}
    if overlap is not None:
        s, e = overlap['start'], overlap['end']
        if s == es:                                   # shares genomic-left boundary
            kind = "alt_5ss" if strand == '+' else "alt_3ss"
        elif e == ee:                                 # shares genomic-right boundary
            kind = "alt_3ss" if strand == '+' else "alt_5ss"
        else:
            kind = "overlapping_exon"
        return "exon_diff_boundary", {'start': s, 'end': e, 'kind': kind}
    if has_up and has_down and not exonic_overlap:
        return "exon_skipped", None
    return "locus_not_covered", None

def map_exon_to_protein(es, ee, cds_obj, strict=True):
    """Returns the CDS/aa annotation dict, or None if the transcript is
    noncoding or the exon falls in UTR. CDS rows MUST be in translation order."""
    cds = _cds_rows(cds_obj)
    if cds is None:
        return None                                   # noncoding transcript
    overlapping = next((c for c in cds if c['end'] >= es and c['start'] <= ee), None)
    if overlapping is None:
        return None                                   # exon is UTR here
    exon_cds_start, exon_cds_end = overlapping['start'], overlapping['end']
 
    cds_offset, gtf_frame = 0, None
    for c in cds:                                     # translation order
        if c['start'] == exon_cds_start and c['end'] == exon_cds_end:
            gtf_frame = c['frame']
            break
        cds_offset += c['end'] - c['start'] + 1
    rel_exon_start = cds_offset
    rel_exon_end = cds_offset + (exon_cds_end - exon_cds_start)
 
    first_frame = cds[0]['frame']                     # row 0 == translation-start CDS
    this_cds_frame = (first_frame - rel_exon_start) % 3
    if this_cds_frame != gtf_frame:
        msg = f"frame mismatch: computed {this_cds_frame}, GTF {gtf_frame} (check CDS ordering)"
        if strict:
            raise AssertionError(msg)
        return {'error': msg}
 
    coding_nt_length = exon_cds_end - exon_cds_start + 1
    overlap_type = ("fully_coding" if exon_cds_start == es and exon_cds_end == ee
                    else "partially_coding" if exon_cds_start >= es and exon_cds_end <= ee
                    else "unexpected")
    return {
        'aa_start': rel_exon_start // 3,
        'aa_end': rel_exon_end // 3,
        'coding_nt_length': coding_nt_length,
        'overlap_type': overlap_type,
        'gtf_frame': gtf_frame,
        'clean_start': gtf_frame == 0,
        'clean_end': (coding_nt_length - gtf_frame) % 3 == 0,
        'frame_preserving': coding_nt_length % 3 == 0,
        
    }
 
def _exon_rows(obj):
    if obj is None:
        return []
    if hasattr(obj, "iterrows"):
        return [{'start': int(r['start']), 'end': int(r['end'])} for _, r in obj.iterrows()]
    return [{'start': int(e['start']), 'end': int(e['end'])} for e in obj]

def annotate_event(event, transcripts_by_gene, exons_by_transcript, cds_by_transcript,
                   strict=True):
    """event: {'chrom','strand','es','ee','gene','us_intron_start','ds_intron_end'}."""
    rec = {'meta': dict(event), 'cluster_id': None,
           'compatible': {}, 'exon_diff_junction': {},
           'exon_diff_boundary': {}, 'exon_skipped': {}, 'locus_not_covered': {}}
    es, ee = event['es'], event['ee']
    us, ds = event['us_intron_start'], event['ds_intron_end']
    
    for _, row in transcripts_by_gene.get(event['gene'], []).iterrows():
        t = row['transcript']
        ttype = row.get('transcript_type') 
        ttag = row.get('tag')
        exons = _exon_rows(exons_by_transcript.get(t))
        if not exons:
            continue
        
        bucket, detail = classify_isoform_detailed(exons, es, ee, us, ds, event['strand']) 
        if bucket in ("compatible", "exon_diff_junction"):
            mapped = map_exon_to_protein(es, ee, cds_by_transcript.get(t), strict=strict)
            entry = mapped if mapped is not None else {'overlap_type': 'noncoding_or_utr'}
        elif bucket == "exon_diff_boundary":
            entry = dict(detail)
            sib_es, sib_ee = detail['start'], detail['end']
            mapped = map_exon_to_protein(sib_es, sib_ee, cds_by_transcript.get(t), strict=strict)
            if mapped is not None:
                entry.update(mapped)
            else:
                entry['overlap_type'] = 'noncoding_or_utr'
        else:
            entry = {}
            
        entry['transcript_type'] = ttype
        entry['transcript_tag'] = ttag
        rec[bucket][t] = entry
        
    return rec

def _cds_rows(obj):
    if obj is None:
        return None
    if hasattr(obj, "iterrows"):
        return [{'start': int(r['start']), 'end': int(r['end']), 'frame': int(r['frame'])}
                for _, r in obj.iterrows()]
    return [{'start': int(c['start']), 'end': int(c['end']), 'frame': int(c['frame'])} for c in obj]

def cluster_events(events):
    """events: list of dicts with chrom,strand,es,ee; adds 'cluster_id' in place."""
    by_csg = defaultdict(list)
    for ev in events:
        by_csg[(ev['chrom'], ev['strand'], ev['gene'])].append(ev)
    cid = 0
    for grp in by_csg.values():
        grp.sort(key=lambda x: (x['es'], x['ee']))
        cur_end = None
        for ev in grp:
            if cur_end is None or ev['es'] > cur_end:
                cid += 1
                cur_end = ev['ee']
            else:
                cur_end = max(cur_end, ev['ee'])
            ev['cluster_id'] = cid
    return events

def mark_sibling_variants(event_info, event_coords):
    """For each variant, flag whether its (start,end) matches
    a *called event* in the same cluster (i.e. the sibling was itself emitted)."""
    coords_by_cluster = defaultdict(set)
    for ev, cid in event_coords.items():
        coords_by_cluster[cid[0]].add((cid[1], cid[2]))   # cid = (cluster_id, es, ee)
    
    # all transcripts that appear as compatible in any event
    compatible_transcripts = set()
    for rec in event_info.values():
        compatible_transcripts.update(rec['compatible'].keys())
        
    for rec in event_info.values():
        cid = rec['cluster_id']
        cluster = coords_by_cluster.get(cid, set())
        parent_es = rec['meta']['es']
        parent_ee = rec['meta']['ee']
        
        # boundary variants: same cluster + shares at least one exon boundary
        for t, sib in rec['exon_diff_boundary'].items():
            sib_es, sib_ee = sib['start'], sib['end']
            sib['is_called_sibling'] = (
                (sib_es, sib_ee) in cluster
                and (sib_es == parent_es or sib_ee == parent_ee)
            )
            
        # junction variants: same exon coords, different flanking partner
        for t, d in rec['exon_diff_junction'].items():
            if not isinstance(d, dict):
                continue
            d['is_called_sibling'] = (parent_es, parent_ee) in cluster
            
        # skipped variants
        for t, d in rec['exon_skipped'].items():
            if not isinstance(d, dict):
                continue
            d['is_called_sibling'] = t in compatible_transcripts
            
    return event_info

In [ ]:
# Load GTF
gtf = pickle.load(open("data/gencode.v50.annotation_gtf_parsed.pkl", "rb"))
gtf_gene = gtf[gtf.feature == "gene"]

gtf_exon = gtf[gtf.feature == "exon"]
gtf_indexed = gtf_exon.set_index(['chrom', 'start', 'end']).sort_index() # for exon lookup
exons_by_transcript = {t: grp for t, grp in gtf_exon.groupby('transcript')}  # for transcript lookup
pickle.dump(exons_by_transcript, open('data/gencode.v46.annotation_exons_by_transcript.pkl', 'wb'))

gtf_cds = gtf[gtf.feature == "CDS"]
cds_by_transcript = {t: grp for t, grp in gtf_cds.groupby('transcript')}  # for transcript lookup for coding sequences
pickle.dump(cds_by_transcript, open('data/gencode.v46.annotation_cds_by_transcript.pkl', 'wb'))

gtf_transcript = gtf[gtf.feature == "transcript"]
transcripts_by_gene = {t: grp for t, grp in gtf_transcript.groupby('gene_name')}  # for transcript lookup for coding sequences
pickle.dump(transcripts_by_gene, open('data/gencode.v46.annotation_transcripts_by_gene.pkl', 'wb'))

KeyError: 'gene'

In [ ]:
exons_by_transcript = pickle.load(open("data/exons_by_transcript.pkl", "rb"))
cds_by_transcript = pickle.load(open("data/cds_by_transcript.pkl", "rb"))
transcripts_by_gene = pickle.load(open("data/transcripts_by_gene.pkl", "rb"))

In [ ]:
SE_anno_df = pickle.load(open("data/SEs_annotated.pkl", "rb"))

event_lookup = {}
for idx, row in SE_anno_df.iterrows():
    if idx not in event_lookup:  # skip if already seen
        event_lookup[idx] = {
            "gene": row['gene_name'],
            "chr": row['chr'],
            "strand": row['strand'],
            "intron_end": row['intron_end'],
            "exon_start": row['exon_start'],
            "exon_end": row['exon_end'],
            "intron_start": row['intron_start'],
            "exon_len": row['exon_len'],
            "exon_coords": row['exon_coords']
        }

In [6]:
len(event_lookup)

47058

# For each event: catalogue all transcript scenarios (same jxns, exon skipped, same exon but different jxns, one shared jxn, etc.)
### Will need this info later to interpret functional consequence of a given splicing event

In [ ]:
# note: this takes ~15 minutes

event_info = {}
event_dicts = []
for idx, rec in event_lookup.items():
    event = {
        'gene': rec['gene'], 
        'chrom': rec['chr'], 
        'strand': rec['strand'],
        'es': int(rec['exon_start']), 
        'ee': int(rec['exon_end']),
        'us_intron_start': rec['intron_start'],
        'ds_intron_end': rec['intron_end'],
    }
    event_info[idx] = annotate_event(event, transcripts_by_gene, exons_by_transcript, cds_by_transcript)
    event_dicts.append({'event': idx, **{k: event[k] for k in ('gene','chrom','strand','es','ee','us_intron_start','ds_intron_end')}})

In [29]:
# group events that share junctions
cluster_events(event_dicts) 

# add this info to `event_info`
for ed in event_dicts:
    event_info[ed['event']]['cluster_id'] = ed['cluster_id']
    
# For each exon_diff_boundary, exon_diff_junction, or skipped variant, flag whether its (start,end) matches a *called event* in the same cluster (i.e. the sibling was itself emitted).
event_coords = {ed['event']: (ed['cluster_id'], ed['es'], ed['ee']) for ed in event_dicts}
event_info = mark_sibling_variants(event_info, event_coords)

In [31]:
pickle.dump(event_dicts, open('data/event_dicts.pkl', 'wb'))
pickle.dump(event_info, open('data/event_info.pkl', 'wb'))

# Ignore

## Log which transcripts were detected in RNA-seq data

In [ ]:
rsem_expr = pd.read_csv("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/bulk/GTEx/cortex/GTEx_cortex_RSEM_TPM.csv", index_col=0)
rsem_expr_subset = rsem_expr[rsem_expr.index.isin(event_info_df['transcript'])]

### Calc isoform vs. ME corr

In [24]:
ctype_abund_df = pd.read_csv("data/ctype_abundance/GTEx_cortex_counts_TMMF_All_501_outliers_removed_top_Qval_mods_PC1_ctype_abundance_filtered_47840genes_cleaned_44846genes_cleaned_mergeParam0.85_subsetCutoff1.427_Modules_top_corr_enriched_w_Claude_marker_genes_PC1_ctype_abundance.csv", index_col=0)
ctype_abund_df.index = ctype_abund_df.index.str.replace(".", "-")

In [25]:
rsem_corr_results = {}
for ct in ctype_abund_df.columns:
    print(ct)
    rsem_corr_results[ct] = rsem_expr_subset.T.corrwith(ctype_abund_df[ct])

CGE Class
All GABAergic
Deep layer glutamatergic
All Neuronal
Oligo
Endo
Peri
OPC
Astro
Micro/PVM
VLMC
Upper layer glutamatergic


In [ ]:
rsem_corr_df = pd.DataFrame(rsem_corr_results)
rsem_corr_df.to_csv(f"data/corrs/GTEx_RSEM_TPM_ctype_abundance_corr.csv")

### Also save isoform mean expression

In [ ]:
rsem_mean_expr = rsem_expr_subset.iloc[:, 1:].mean(axis=1)
is_detected = rsem_mean_expr.index.isin(event_info_df.transcript)

In [28]:
rsem_info_df = pd.DataFrame({'RSEM_detected': is_detected, 'RSEM_mean_expr': rsem_mean_expr}, index=rsem_mean_expr.index)

## Now append exon info. to cell type exon analysis results

In [29]:
pd.set_option('display.max_columns', None)

In [30]:
column_order = ['Gene', 'is_specific', 'specific_direction', 
                'chr', 'strand', 'exon_start', 'exon_end', 'exon_len', 'transcript', 'transcript_type', 
                'exon_number', 'tag', 'coding_nt_length', 'full_exon_nt_length', 'overlap_type',  
                'aa_start', 'aa_end', 'RSEM_detected', 'RSEM_mean_expr', 'RSEM_expr_corr', 
                'r', 'fdr', 
                'CGE Class', 'All GABAergic', 'All Neuronal',
                'Upper layer glutamatergic', 'Deep layer glutamatergic', 'Oligo', 'OPC',
                'Astro', 'Micro/PVM', 'VLMC', 'Endo', 'Peri'
                ]

In [31]:
rsem_corr_df.columns = rsem_corr_df.columns.str.replace("/", "_").str.replace(" ", "_")

In [ ]:
for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        ct = file.split("_exons.csv")[0]
        print(ct)
        
        signif_exons_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
        
        ctype_corr = rsem_corr_df[ct]
        ctype_corr.name = "RSEM_expr_corr"
        rsem_corr_info_df = rsem_info_df.merge(
            ctype_corr, left_index=True, right_index=True
        )
        exon_rsem_info_df = event_info_df.merge(
            rsem_corr_info_df, left_on="transcript", right_index=True, how="left"
        )
        mask = exon_rsem_info_df.RSEM_detected == True 
        exon_rsem_info_df.loc[~mask, 'RSEM_detected'] = False
        
        signif_exons_info = signif_exons_df.merge(
            exon_rsem_info_df, 
            left_index=True, 
            right_index=True,
            how='left'
        )
        rest_columns = signif_exons_info.columns[signif_exons_info.columns.str.contains("diff")].tolist() 
        new_file = file.replace('_exons.csv', '_exons_annotated.csv')
        signif_exons_info[column_order + rest_columns].to_csv(f"data/ctype_exons/annotated/{new_file}")

Oligo
VLMC
Endo
Deep_layer_glutamatergic
Astro
OPC
Micro_PVM
All_Neuronal
All_GABAergic
Peri
CGE_Class
Upper_layer_glutamatergic


In [41]:
pd.set_option('display.max_columns', None)